<a href="https://colab.research.google.com/github/shreyasat27/mahework2025/blob/main/DiVin_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Complete one file to prove DiVincenzo's Criteria for "Well defined two level system"**

In [ ]:
!pip install pyscf
!pip install rdkit
!pip install qutip

In [ ]:
def analyze_molecule(atom_geometry, basis="sto-3g", charge=0, spin=0):
    mol = gto.Mole()
    mol.atom = atom_geometry
    mol.basis = basis
    mol.charge = charge
    mol.spin = spin
    mol.build()

    if spin == 0:
        mf = scf.RHF(mol)
    else:
        mf = scf.UHF(mol)

    # Improve convergence
    mf.max_cycle = 100
    mf.init_guess = 'atom'
    mf.damp = 0.5
    mf.level_shift = 0.2

    energy = mf.kernel()

    # Check convergence
    if not mf.converged:
        print("WARNING: SCF did not fully converge!")

    mo_energy = mf.mo_energy
    homo = np.max(mo_energy[mo_energy < 0])
    lumo = np.min(mo_energy[mo_energy > 0])
    gap = lumo - homo

    S2, multiplicity = mf.spin_square()

    print("=== Molecule Analysis ===")
    print(f"Total Energy: {energy:.6f} Hartree")
    print(f"HOMO Energy: {homo:.6f} Ha")
    print(f"LUMO Energy: {lumo:.6f} Ha")
    print(f"HOMO-LUMO Gap: {gap:.6f} Ha")
    print(f"Spin <S^2>: {S2:.4f}")
    print(f"Spin Multiplicity: {multiplicity:.4f}")

    # Strict qubit criteria:
    # 1. Multiplicity ≈ 2 (doublet)
    # 2. <S²> ≈ 0.75 (pure spin-1/2)
    # 3. Gap > 0
    if (abs(multiplicity - 2) < 0.1 and abs(S2 - 0.75) < 0.1 and gap > 0):
        print("\n✅ Candidate qubit system (well-defined two-level + spin-1/2).")
    else:
        print("\n⚠️ Not an ideal qubit candidate (check gap or spin state).")

    return {
        "energy": energy,
        "homo": homo,
        "lumo": lumo,
        "gap": gap,
        "S2": S2,
        "multiplicity": multiplicity
    }

In [ ]:
result = analyze_molecule("C 0 0 0; H 1.0 0 0; H -0.5 0.866 0; H -0.5 -0.866 0", spin =1)

In [ ]:
# Re-run NO with improved settings
result = analyze_molecule("N 0 0 0; O 1.15 0 0", spin=1)

In [ ]:
result = analyze_molecule(
    "O 0 0 0; H 0 0 0.96; H 0 0.96 0",  # H₂O geometry (bond angle ~104.5°)
    spin=0,  # Singlet (no unpaired electrons)
    charge=0,
    basis="sto-3g"
)